Step 1: Install Dependency

In [ ]:
%pip install -q kagglehub

from pathlib import Path

import cv2
import kagglehub
import numpy as np
import pandas as pd
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedShuffleSplit, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier


Step 2: Download data from kaggle
I use downloaded zip on my PC, this black is not very important

In [ ]:
path = kagglehub.dataset_download("gpreda/chinese-mnist")

def resolve_chinese_mnist_paths(dataset_root):
    root = Path(dataset_root)
    csv_matches = sorted(root.rglob("chinese_mnist.csv"))
    if not csv_matches:
        raise FileNotFoundError(f"No chinese_mnist.csv under {root}")
    csv_path = csv_matches[0]
    img_dir = csv_path.parent / "data" / "data"
    if not img_dir.is_dir():
        jpgs = list(root.rglob("input_*.jpg"))
        if not jpgs:
            raise FileNotFoundError("No image folder with input_*.jpg")
        img_dir = jpgs[0].parent
    return csv_path, img_dir


DATASET_ROOT = Path(path)
CSV_PATH, IMG_DIR = resolve_chinese_mnist_paths(DATASET_ROOT)
print(CSV_PATH)
print(IMG_DIR)

df_all = pd.read_csv(CSV_PATH)
label_col = "code"
df_all["path"] = df_all.apply(
    lambda r: IMG_DIR / f"input_{int(r.suite_id)}_{int(r.sample_id)}_{int(r.code)}.jpg",
    axis=1,
)

Step 3: Get subset for train and test

In [ ]:
train_pool_df, test_df = train_test_split(
    df_all,
    test_size=1000,
    stratify=df_all[label_col],
    random_state=42,
)
train_pool_df = train_pool_df.reset_index(drop=True)

placeholder = np.zeros((len(train_pool_df), 1))
y_pool = train_pool_df[label_col].to_numpy()

splitter_5k = StratifiedShuffleSplit(
    n_splits=1, train_size=5000, random_state=42
)
for train_idx, i in splitter_5k.split(placeholder, y_pool):
    train_5k_df = train_pool_df.iloc[train_idx].reset_index(drop=True)

splitter_10k = StratifiedShuffleSplit(
    n_splits=1, train_size=10000, random_state=42
)
for train_idx, i in splitter_10k.split(placeholder, y_pool):
    train_10k_df = train_pool_df.iloc[train_idx].reset_index(drop=True)

Step 4: Reshape

In [ ]:
X_list, y_list = [], []
for i, row in test_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_test = np.asarray(X_list, dtype=np.float32)
y_test = np.asarray(y_list)

X_list, y_list = [], []
for i, row in train_5k_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_train_5k = np.asarray(X_list, dtype=np.float32)
y_train_5k = np.asarray(y_list)

X_list, y_list = [], []
for i, row in train_10k_df.iterrows():
    img = cv2.imread(str(row["path"]), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(row["path"])
    X_list.append(img.reshape(-1))
    y_list.append(int(row[label_col]))
X_train_10k = np.asarray(X_list, dtype=np.float32)
y_train_10k = np.asarray(y_list)

LABELS = np.sort(np.unique(y_test))

Step 5&6: init and train


In [ ]:
fits = []
for Xtr, ytr in (X_train_5k, y_train_5k), (X_train_10k, y_train_10k):
    fits.append(
        (
            KNeighborsClassifier(n_neighbors=3).fit(Xtr, ytr),
            DecisionTreeClassifier().fit(Xtr, ytr),
            SGDClassifier(max_iter=250).fit(Xtr, ytr),
        )
    )
(knn_5k, dt_5k, sgd_5k), (knn_10k, dt_10k, sgd_10k) = fits
for n in (5000, 10000):
    print(n, "done")


Step 7&8: Predit and Evaluation

In [ ]:
print("acc, prec, rec, f1")
for n_train, knn, dt, sgd in (
    (5000, knn_5k, dt_5k, sgd_5k),
    (10000, knn_10k, dt_10k, sgd_10k),
):
    print("Dataset", n_train)
    for name, clf in (("KNN", knn), ("DT", dt), ("SGD", sgd)):
        pred = clf.predict(X_test)
        print(
            name,
            accuracy_score(y_test, pred),
            precision_score(y_test, pred, average="macro", zero_division=0),
            recall_score(y_test, pred, average="macro", zero_division=0),
            f1_score(y_test, pred, average="macro", zero_division=0),
        )
        print(confusion_matrix(y_test, pred, labels=LABELS))
    print()
